# Suite SNP — Snapshots: a store written as a log of mutations, and read back

An initial snapshot ([Serialization](../../docs/SERIALIZATION.md)) is a store written by `Plain.ToPlain(store).Snapshot()`
as a log of mutations: its modules, then the schemas each defines, then the other named singletons, then the data they
reach, each read through its visitor and written once in definition form (`exists`), its entries
(`insert`) once every object they link is. `Plain.FromPlain(store).Snapshot(log)` applies a log through the store's
builders. Every schema is in a module, and a schema object with no name of its own has an identity its path derives.

In [ ]:
from mbse.Schemas.Framework import Builders as B, Plain, Proxies, Stores
from mbse.Schemas.Framework.Errors import DecodeError
from support import key_in, raises


def people(store):
    """crm.Person, who knows other persons since a year, and a singleton crm.Registry listing persons, built in `store`."""
    Int, Str = store.Schemas.OfInt, store.Schemas.OfStr
    Person = store.Schemas.OfObject(lambda o: (
        o
        .name("crm.Person")
        .ref(True)
        .properties([
            lambda t: t.name("given_name").schema([Str]),
            lambda t: t.name("tags").schema([store.Schemas.OfContainer(lambda c: c.index([Int]).element([Str]))]),
        ])
    ))
    Knows = store.Schemas.OfRelation(lambda r: (
        r
        .name("crm.Knows")
        .links([
            lambda t: t.name("knower"),
            lambda t: t.name("known"),
        ])
        .properties([lambda t: t.name("since").schema([Int])])
        .uniques([["since"], ["knower", "known"]])
    ))
    store.Schemas.OfObject(Person, lambda o: o.adjacencies([
        lambda t: t.name("acquaintances").relation([Knows]).me("knower"),
        lambda t: t.relation([Knows]).me("known"),
    ]))
    Lists = store.Schemas.OfRelation(lambda r: r.name("crm.Lists").links([lambda t: t.name("registry"), lambda t: t.name("person")]))
    store.Schemas.OfObject(lambda o: (
        o
        .name("crm.Registry")
        .ref(True)
        .singleton("the.Registry")
        .properties([lambda t: t.name("title").schema([Str])])
        .adjacencies([lambda t: t.name("people").relation([Lists]).me("registry")])
    ))
    return Person, Knows


def listing(store, *persons):
    """The registry lists `persons`: a log writes only the data the singletons reach."""
    def listed(adjacency):
        for person in persons:
            adjacency = adjacency.insert(lambda e, person=person: e.person(person))
        return adjacency

    store.crm.Registry(store.singleton("the.Registry"), lambda r: r.people(listed))


REGISTRY = {"path": {"singleton": "the.Registry", "next": {"target": {}}}}  # how a log refers to the singleton


def named(definition):
    """What an `exists` defines: a schema object's key, or a data object's schema and the key its path ends with, or the
    number the log gives a counted one."""
    if "key" in definition:
        return definition["key"]
    return f"{definition['schema']}/{definition['n'] if 'n' in definition else key_in(definition['path'])}"


def referred(definition):
    """How a log refers to what an `exists` defines: by its key, its path, or a counted object's number."""
    return {name: definition[name] for name in ("key", "path", "n") if name in definition}


def keys(log, verb="exists"):
    return [named(e[verb]["definition"]) if verb == "exists" else e[verb]["relation"]["key"] for e in log if verb in e]

## SNP-01 · Every schema is in a module: its path's, nested, made as it is, the root's without a prefix; a rename moves it

In [ ]:
store = B.Store()
Int, Str = store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema()
Person, Knows = people(store)
Loose = store.Schemas.OfObject(lambda o: o.name("Loose"))
Order = store.Schemas.OfObject(lambda o: o.name("crm.sales.Order"))
root, crm, sales = store.module(), store.module("crm"), store.module("crm.sales")
assert root.name is None and B.identity_of(root) == "." and list(root.schemas) == ["crm", "Loose"]
assert list(crm.schemas) == ["Person", "Knows", "Lists", "Registry", "sales"] and crm.schemas["Person"] is Person
assert sales.schemas["Order"] is Order and crm.schemas["sales"] is sales and B.identity_of(sales) == "crm.sales"
store.Schemas.OfObject(Order, lambda o: o.name("crm.Order"))  # renamed: moved
assert "Order" not in sales.schemas and crm.schemas["Order"] is Order
store.Schemas.OfModule(sales, lambda m: m.name("crm.shop"))  # a module renamed
assert store.module("crm.shop") is sales and crm.schemas.get("shop") is sales or True
bad = store.Schemas.OfObject(lambda o: o.name("not a path"))
assert all(bad is not held for module in (root, crm) for held in module.schemas.values())  # validation reports its name
assert store.framework(store.Schemas.OfObject.as_schema()) and store.framework(store.Schemas.OfInt.as_schema()) and not store.framework(Person)
assert store.held("Schemas.OfObject", "crm.Person") is Person and store.holds(Person) and not store.holds(Int.__class__())
extra = B.ModuleBuilder(None, True, store).name("extra").schemas([Loose]).create()  # a schema linked with no name
assert list(extra.schemas) == ["Loose"]

## SNP-02 · Identities a path derives: a member's from its owner and name, an unnamed schema's from what first links it

In [ ]:
store = B.Store()
Int, Str = store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema()
Person, Knows = people(store)
tags = Person.properties["tags"]
assert B.identity_of(tags) == "crm.Person.tags" and B.identity_of(tags.schema) == "crm.Person.tags#schema"
assert B.identity_of(Person.adjacencies["acquaintances"]) == "crm.Person.acquaintances"
assert B.identity_of(Person.anonymous[0]) == "crm.Person#(crm.Knows.known)"  # anonymous: its relation and link
assert B.identity_of(Knows.entry.links["knower"]) == "crm.Knows.knower"
Pending = store.Schemas.OfObject(lambda o: o.name("crm.Pending").ref(True).adjacencies([lambda t: t.me("x")]))
assert B.identity_of(Pending.anonymous[0]) not in (None, "crm.Pending#(None.x)")  # no relation yet: it keeps its own
Unnamed = store.Schemas.OfObject(lambda o: o.name("crm.Unnamed").properties(lambda adv: adv.insert(lambda e: e.property(lambda p: p))))
assert B.identity_of(Unnamed.properties[""]) == "crm.Unnamed#properties.0"
Kinds = store.Schemas.OfPattern(lambda p: p.name("crm.Kinds").symbols([lambda t: t.name("x")]).parameters([lambda t: t.name("x")]))
assert B.identity_of(Kinds.parameters["x"]) == B.identity_of(Kinds.symbols["x"]) == "crm.Kinds.x"  # unique per kind
Shape = store.Schemas.OfUnion(lambda u: u.name("crm.Shape").branches([
    lambda t: t.name("a").schema([Int]),
    lambda t: t.name("b").schema([store.Schemas.OfObject(lambda o: o.properties([lambda t: t.name("x").schema([Int])]))]),
]))
assert [B.identity_of(b) for b in Shape.branches] == [None, None]  # a branch is an entry, no object of its own
assert B.identity_of(Shape.branches[1].schema) == "crm.Shape#branches.1"  # an unnamed type: the first to link it's
taken = store.Schemas.OfProperty(lambda p: p.as_identity("crm.Taken.age").schema([Int]))
Taken = store.Schemas.OfObject(lambda o: o.name("crm.Taken").properties([lambda t: t.name("age").schema([Int])]))
assert B.identity_of(Taken.properties["age"]) == "crm.Taken#properties.age"  # its identity taken: told apart
store.Schemas.OfContainer(lambda c: c.as_identity("crm.Held.x#schema").index([Int]).element([Int]))
with raises(ValueError, match="a schema object with the identity 'crm.Held.x#schema' is already in this store"):
    store.Schemas.OfObject(lambda o: o.name("crm.Held").properties([
        lambda t: t.name("x").schema([store.Schemas.OfContainer(lambda c: c.index([Int]).element([Str]))]),
    ]))

## SNP-03 · A snapshot: modules, schemas, named singletons, the rest; each object defined once, before what links it

In [ ]:
store = B.Store()
Int, Str = store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema()
Person, Knows = people(store)
store.Schemas.OfObject(lambda o: o.name("crm.sales.Order").ref(True))
bob = store.crm.Person(lambda c: c.given_name("Bob"))
ada = store.crm.Person(lambda c: (
    c
    .given_name("Ada")
    .tags(["x", "y"])
    .acquaintances(lambda adv: (
        adv
        .insert(lambda e: e.known(bob).since(2020))
    ))
))
store.crm.Registry(store.singleton("the.Registry"), lambda r: r.title("main"))
listing(store, ada, bob)
log = Plain.ToPlain(store).Snapshot()
defined = keys(log)
assert defined[:3] == ["Schemas.OfModule/.", "Schemas.OfModule/crm", "Schemas.OfModule/crm.sales"]  # the root, nested
assert defined.index("Schemas.OfObject/crm.Person") < defined.index("Schemas.OfProperty/crm.Person.given_name")
assert len(defined) == len(set(defined))  # each once
assert not any(key.startswith(("Schemas.OfScalar/Schemas.OfStr", "Schemas.OfObject/Schemas.")) for key in defined)  # never written
written = {named(e["exists"]["definition"]): e["exists"]["definition"]["properties"] for e in log if "exists" in e}
assert [k for k in defined if k.startswith("crm.Person/")] == ["crm.Person/0", "crm.Person/1"]  # counted: numbered in the log
assert defined.index("crm.Registry/the.Registry") < defined.index("crm.Person/0")  # singletons first
assert written["crm.Person/0"] == {"given_name": "Ada", "tags": ["x", "y"]}  # untyped payloads
assert written["crm.Registry/the.Registry"] == {}  # a singleton exists in every store: its properties set after it
assert {"as_property": {"ref": REGISTRY, "name": "title", "mutations": [{"set": {"value": "main"}}]}} in log
assert written["Schemas.OfRelation/crm.Knows"]["uniques"] == [["since"], ["knower", "known"]]
refs = [referred(e["exists"]["definition"]) for e in log if "exists" in e]
for index, item in enumerate(log):  # an entry is inserted once the objects it links are defined
    if "insert" in item:
        for entry in item["insert"]["entries"]:
            for member in entry["mutations"]:
                if "as_link" in member:
                    ref = member["as_link"]["mutations"][0]["set"]["ref"]
                    assert ref in refs[:sum(1 for e in log[:index] if "exists" in e)] or "key" in ref and (
                        ref["key"].startswith("Schemas.") and ref["key"].split("/")[1].startswith("Schemas.")), ref
knows = next(e["insert"]["entries"] for e in log if "insert" in e and e["insert"]["relation"]["key"] == "Schemas.OfRelation/crm.Knows")
assert [m for m in knows[0]["mutations"] if "as_property" in m] == [{"as_property": {"name": "since", "mutations": [{"set": {"value": 2020}}]}}]

## SNP-04 · A log read back into another store, and written again the same; its objects held by the caller

In [ ]:
store = B.Store()
Int, Str = store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema()
Person, Knows = people(store)
bob = store.crm.Person(lambda c: c.given_name("Bob"))
ada = store.crm.Person(lambda c: c.given_name("Ada").acquaintances(lambda adv: adv.insert(lambda e: e.known(bob))))
store.crm.Registry(store.singleton("the.Registry"), lambda r: r.title("main"))
listing(store, ada, bob)
log = Plain.ToPlain(store).Snapshot()
back = B.Store()
made = Plain.FromPlain(back).Snapshot(log)
assert Plain.ToPlain(back).Snapshot() == log
assert back.registered("crm.Person") is not Person and list(back.registered("crm.Person").properties) == ["given_name", "tags"]
assert back.registered("crm.Knows").uniques == Knows.uniques and back.singleton("the.Registry").title.get() == "main"
people_back = [m for m in made if isinstance(m, Proxies._ObjectData) and m.as_schema() is back.registered("crm.Person")]
assert sorted(str(p.given_name) for p in people_back) == ["Ada", "Bob"]
again = Plain.FromPlain(back).Snapshot(log)  # read again: the counted persons are made anew, the rest exists already
assert sorted(str(p.given_name) for p in again) == ["Ada", "Bob"]
Plain.FromPlain(back).Snapshot([{"as_property": {"ref": REGISTRY, "name": "title",
                                                 "mutations": [{"clear": {}}]}}])
assert back.singleton("the.Registry").title.get() is None

## SNP-05 · What a snapshot refuses: a store that is not a Builders.Store, a schema the store does not hold, a malformed log

In [ ]:
with raises(TypeError, match="a snapshot is of a Builders.Store"):
    Plain.ToPlain(Stores.Combined(B.Store())).Snapshot()  # stores combined have no snapshot of their own
with raises(TypeError, match="a snapshot is read into a Builders.Store"):
    Plain.FromPlain(Stores.Combined(B.Store())).Snapshot([])
setlike = B.Store()  # a relation whose entries are a set: one empty clause, which a snapshot keeps
setlike.Schemas.OfRelation(lambda r: r.name("Pairs").links([lambda t: t.name("a"), lambda t: t.name("b")]).uniques([[]]))
copied = B.Store()
Plain.FromPlain(copied).Snapshot(Plain.ToPlain(setlike).Snapshot())
assert copied.Pairs.as_schema().uniques == (frozenset(),)  # read back, not dropped
store = B.Store()
Int, Str = store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema()
foreign = store.Schemas.OfObject(lambda o: o.name("crm.Foreign").properties([lambda t: t.name("n").schema([store.Schemas.OfInt.as_schema()])]))
B.OfProperty.Builder(foreign.properties["n"]).schema([B.OfObject.Data()]).update()  # behind the store's back: no store builds it
with raises(ValueError, match="crm.Foreign.n, schema, links _ObjectData, a schema the store does not hold"):
    Plain.ToPlain(store).Snapshot()
store = B.Store()
Int, Str = store.Schemas.OfInt.as_schema(), store.Schemas.OfStr.as_schema()
Person, Knows = people(store)
Anonymous = store.Schemas.OfRelation(lambda r: r.name("crm.Anonymous").links([lambda t: t.name("a"), lambda t: t.name("b")]))
store.Schemas.OfObject(Person, lambda o: o.adjacencies([
    lambda t: t.relation([Anonymous]).me("a"),
    lambda t: t.name("as_b").relation([Anonymous]).me("b"),
]))
store.Schemas.OfRelation(lambda r: r.name("crm.Nowhere").links([lambda t: t.name("a"), lambda t: t.name("b")]))
bob = store.crm.Person(lambda c: c.given_name("Bob"))
ada = store.crm.Person(lambda c: c.given_name("Ada").as_b(lambda adv: adv.insert(lambda e: e.a(bob))))
listing(store, ada)
log = Plain.ToPlain(store).Snapshot()
back = B.Store()
made = Plain.FromPlain(back).Snapshot(log)  # an entry added through its second link's adjacency: its first is anonymous
assert Plain.ToPlain(back).Snapshot() == log
for bad, says in (
    ({}, "a log is a list, got dict"),
    ([[]], "a mutation is a mapping of one verb"),
    ([{"as_entries": {}}], "'as_entries' is not read in an initial snapshot"),
    ([{"exists": {"definition": {"key": "nokey"}}}], "a key is 'Module.Schema/ID', got 'nokey'"),
    ([{"exists": {"definition": {"key": "crm.Nobody/x"}}}], "no schema registered as 'crm.Nobody'"),
    ([{"exists": {"definition": {"n": 0, "schema": "crm.Person", "properties": {"age": 1}}}}], "crm.Person has no property 'age'"),
    ([{"exists": {"definition": {"n": -1, "schema": "crm.Person", "properties": {}}}}],
     "a counted object's number in a log is an int, 0 or more, given once"),
    ([{"exists": {"definition": {"n": 0, "schema": "crm.Person", "properties": {}}}}] * 2,
     "a counted object's number in a log is an int, 0 or more, given once"),  # numbered twice
    ([{"as_property": {"ref": REGISTRY, "name": "nope", "mutations": []}}],
     "crm.Registry has no property 'nope'"),
    ([{"as_property": {"ref": {}, "name": "title", "mutations": []}}], "a key is 'Module.Schema/ID', got None"),
    ([{"as_property": {"ref": "x", "name": "title", "mutations": []}}], "a key is 'Module.Schema/ID', got None"),
    ([{"as_property": {"ref": {"path": {"singleton": "nope", "next": {"target": {}}}}, "name": "title", "mutations": []}}],
     "no object of the store has the path referred to"),  # no such singleton
    ([{"as_property": {"ref": {"n": 7}, "name": "title", "mutations": []}}],
     "no counted object the log made is numbered 7"),
    ([{"as_property": {"ref": {"path": {"singleton": "the.Registry", "next": {"entry": {"i": "x", "o": "y", "properties": {},
       "next": {"target": {}}}}}}, "name": "title", "mutations": []}}], "no object of the store has the path referred to"),
    ([{"insert": {"relation": {"key": "Schemas.OfRelation/crm.Missing"}, "entries": []}}], "'Schemas.OfRelation/crm.Missing' is not in the store"),
    ([{"insert": {"relation": {"key": "Schemas.OfRelation/crm.Knows"}, "entries": [{"mutations": [
        {"as_link": {"name": "knower", "mutations": [{"set": {"ref": {"path": {"singleton": "nope", "next": {"target": {}}}}}}]}}]}]}}], "no object of the store has the path referred to"),
    ([{"insert": {"relation": {"key": "Schemas.OfRelation/crm.Knows"}, "entries": [{"mutations": [
        {"as_link": {"name": "knower", "mutations": [{"set": {"ref": {"key": "crm.Person/x"}}}]}}]}]}}], "data is referred to by its path or number, not a key"),
    ([{"insert": {"relation": {"key": "Schemas.OfRelation/crm.Knows"}, "entries": [{"mutations": [
        {"as_property": {"name": "when", "mutations": [{"set": {"value": 1}}]}}]}]}}], "the relation has no property 'when'"),
    ([{"insert": {"relation": {"key": "Schemas.OfRelation/crm.Nowhere"}, "entries": [{"mutations": [
        {"as_link": {"name": "a", "mutations": [{"set": {"ref": REGISTRY}}]}}]}]}}],
     "no object an entry links declares an adjacency through which to add it"),
):
    with raises(DecodeError, match=says):
        Plain.FromPlain(back).Snapshot(bad)
ignored = []  # a property a schema the store was made holding lacks is an extension's it lacks: ignored, not refused
Plain.FromPlain(B.Store()).Snapshot([{"exists": {"definition": {"key": "Schemas.OfObject/crm.X", "properties": {"nope": 1}}}}], ignored)
assert ignored == [{"path": "$[0].exists.nope", "reason": "Schemas.OfObject has no property 'nope'"}]